# FASTopic - Notebook PPD

Pipeline aligned with ECTRM datasets and `.mat` outputs.


## 1) Install dependencies (Kaggle-safe)


In [15]:
import sys
import subprocess
import importlib.util
import pandas as pd


def ensure_package(import_name: str, pip_spec: str, no_deps: bool = False):
    if importlib.util.find_spec(import_name) is None:
        cmd = [
            sys.executable,
            '-m',
            'pip',
            'install',
            '--upgrade-strategy',
            'only-if-needed',
            '--no-cache-dir',
        ]
        if no_deps:
            cmd.append('--no-deps')
        cmd.append(pip_spec)
        print('Installing', pip_spec)
        subprocess.check_call(cmd)


# Important: we do NOT install/upgrade torch, CUDA, or GPU drivers.
# Kaggle's existing GPU stack is reused as-is.
ensure_package('scipy', 'scipy')
ensure_package('yaml', 'pyyaml')
ensure_package('sentence_transformers', 'sentence-transformers')
ensure_package('fastopic', 'fastopic==1.0.1', no_deps=True)
ensure_package('gensim', 'gensim')

print('Dependency check OK (torch/cuda untouched)')


Dependency check OK (torch/cuda untouched)


In [16]:
print("numpy handled by dependency cell")


numpy handled by dependency cell


## 2) Imports and env


In [17]:
import os
import time
import random
from pathlib import Path

import numpy as np
import scipy
import scipy.io
import scipy.sparse as sp
import torch
import yaml
from sklearn import metrics
from fastopic import FASTopic

print('Python:', __import__('sys').version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


Python: 3.12.12
NumPy: 2.0.2
SciPy: 1.16.3
Torch: 2.9.0+cu126
CUDA available: True


## 3) Dataset discovery

La cellule suivante contient la configuration et la detection automatique des datasets (local/Kaggle).


In [18]:
TARGET_DATASETS = ['20NG', 'AGNews', 'IMDB', 'YahooAnswer']
K_LIST = [20, 50, 100]
RANDOM_SEED = 42

REQUIRED_DATA_FILES = [
    'train_bow.npz', 'test_bow.npz', 'train_labels.txt', 'test_labels.txt', 'vocab.txt', 'word_embeddings.npz'
]

IS_KAGGLE = Path('/kaggle').exists()


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'ECTRM_source' / 'ECRTM' / 'data').exists():
            return candidate
    return start


if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working')
    INPUT_ROOT = Path('/kaggle/input')
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    INPUT_ROOT = PROJECT_ROOT


def has_required_files(path: Path) -> bool:
    return path.exists() and path.is_dir() and all((path / f).exists() for f in REQUIRED_DATA_FILES)


def guess_dataset_name(path: Path):
    s = str(path).lower()
    if '20ng' in s or '20news' in s:
        return '20NG'
    if 'agnews' in s or 'ag_news' in s:
        return 'AGNews'
    if 'yahoo' in s:
        return 'YahooAnswer'
    if 'imdb' in s:
        return 'IMDB'
    return None


def iter_candidate_dirs(base: Path, max_depth: int = 5):
    if not base.exists():
        return
    base_depth = len(base.parts)
    for root, dirs, _ in os.walk(base):
        root_path = Path(root)
        depth = len(root_path.parts) - base_depth
        if depth > max_depth:
            dirs[:] = []
            continue
        yield root_path


def discover_dataset_dirs():
    found = {}

    local_data_root = PROJECT_ROOT / 'ECTRM_source' / 'ECRTM' / 'data'
    for ds in TARGET_DATASETS:
        p = local_data_root / ds
        if has_required_files(p):
            found[ds] = p

    scan_roots = [Path('/kaggle/input'), PROJECT_ROOT, INPUT_ROOT] if IS_KAGGLE else [PROJECT_ROOT, INPUT_ROOT]
    seen = set()
    for scan_root in scan_roots:
        for cand in iter_candidate_dirs(scan_root):
            cs = str(cand)
            if cs in seen:
                continue
            seen.add(cs)
            if not has_required_files(cand):
                continue
            ds = guess_dataset_name(cand)
            if ds is not None and ds not in found:
                found[ds] = cand
    return found


DATASET_DIRS = discover_dataset_dirs()
print('IS_KAGGLE:', IS_KAGGLE)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('INPUT_ROOT:', INPUT_ROOT)
print('\nResolved datasets:')
for ds in TARGET_DATASETS:
    print('-', ds, DATASET_DIRS.get(ds))


IS_KAGGLE: True
PROJECT_ROOT: /kaggle/working
INPUT_ROOT: /kaggle/input

Resolved datasets:
- 20NG /kaggle/input/datasets/snlosnl/kaggleinputectrm-source-data-20ng
- AGNews /kaggle/input/datasets/snlosnl/ectrm-sourcedataagnews
- IMDB /kaggle/input/datasets/snlosnl/ectrm-sourcedataimdb
- YahooAnswer /kaggle/input/datasets/snlosnl/ectrm-yahooanswer


## 4) Config and helpers


In [19]:
OUTPUT_DIR = (Path('/kaggle/working') / 'Sortie_mat' / 'output' / 'kaggle' / 'FASTopic') if IS_KAGGLE else (PROJECT_ROOT / 'Sortie_mat' / 'FASTopic')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR:', OUTPUT_DIR)

# FASTopic-specific defaults (phase 2 tuning).
# Important: we intentionally do NOT reuse ECRTM batch_size=200 for low_memory_batch_size,
# because FASTopic docs warn that too small low_memory_batch_size hurts quality.
FASTOPIC_DEFAULTS = {
    'epochs': 200,
    'learning_rate': 0.002,
    'low_memory': True,
    'normalize_embeddings': True,
    'doc_embed_model': 'all-MiniLM-L6-v2',
    'min_low_memory_batch_size': 2000,
    'max_low_memory_batch_size': 4000,
    'dt_alpha_default': 10.0,
    'tw_alpha': 2.0,
    'theta_temp': 1.0,
}

# Options: 'auto' | 'text' | 'bow_avg'
DOC_EMBEDDING_MODE = 'auto'

# Good default split for your datasets (short texts -> bow_avg, long texts -> text).
DOC_MODE_BY_DATASET = {
    '20NG': 'text',
    'IMDB': 'text',
    'AGNews': 'bow_avg',
    'YahooAnswer': 'bow_avg',
}

# FASTopic README suggests higher DT_alpha when topics are repetitive.
DT_ALPHA_BY_DATASET = {
    '20NG': 8.0,
    'IMDB': 8.0,
    'AGNews': 12.0,
    'YahooAnswer': 12.0,
}


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_cfg(path):
    if path is None or (not Path(path).exists()):
        return {}
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f) or {}


def discover_cfg_paths():
    cfg = {}

    local_cfg_roots = [
        PROJECT_ROOT / 'ECTRM_source' / 'ECRTM' / 'ECRTM' / 'configs' / 'model',
        PROJECT_ROOT / 'ECTRM_source' / 'ECRTM' / 'configs' / 'model',
    ]

    for root in local_cfg_roots:
        for ds in TARGET_DATASETS:
            p = root / f'ECRTM_{ds}.yaml'
            if p.exists():
                cfg[ds] = p

    for ds in TARGET_DATASETS:
        if ds in cfg:
            continue
        name = f'ECRTM_{ds}.yaml'
        for base in [Path('/kaggle/input'), Path('/kaggle/working'), PROJECT_ROOT]:
            if not base.exists():
                continue
            found = None
            for r, _, files in os.walk(base):
                if name in files:
                    found = Path(r) / name
                    break
            if found is not None:
                cfg[ds] = found
                break

    return cfg


CONFIGS = discover_cfg_paths()
for ds in TARGET_DATASETS:
    print(ds, 'cfg:', CONFIGS.get(ds))


def load_vocab(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f if line.strip()]


def load_bow_csr(path: Path):
    return sp.load_npz(path).astype(np.float32).tocsr()


def load_word_embeddings(path: Path, vocab_size=None):
    try:
        arr = sp.load_npz(path).astype(np.float32).toarray()
        if vocab_size is not None and arr.shape[0] != vocab_size and arr.shape[1] == vocab_size:
            arr = arr.T
        return arr
    except Exception:
        data = np.load(path)
        if isinstance(data, np.lib.npyio.NpzFile):
            arr = data[list(data.keys())[0]]
        else:
            arr = data
        arr = np.asarray(arr, dtype=np.float32)
        if vocab_size is not None:
            if arr.ndim == 1:
                emb_dim = arr.size // vocab_size
                arr = arr[: vocab_size * emb_dim].reshape(vocab_size, emb_dim)
            elif arr.shape[0] != vocab_size and arr.shape[1] == vocab_size:
                arr = arr.T
        return arr.astype(np.float32)


def load_texts_if_exists(path: Path):
    if not path.exists():
        return None
    with open(path, 'r', encoding='utf-8') as f:
        return [line.strip() for line in f]


def average_token_length(texts):
    if texts is None or len(texts) == 0:
        return 0.0
    return float(np.mean([len(t.split()) for t in texts]))


def resolve_doc_mode(dataset, train_docs):
    if DOC_EMBEDDING_MODE in ('text', 'bow_avg'):
        return DOC_EMBEDDING_MODE

    # 'auto': dataset-specific rule first, then length heuristic fallback.
    mode = DOC_MODE_BY_DATASET.get(dataset)
    if mode is not None:
        return mode

    avg_len = average_token_length(train_docs)
    return 'text' if avg_len >= 30 else 'bow_avg'


def resolve_low_memory_batch_size(n_docs):
    min_bs = int(FASTOPIC_DEFAULTS['min_low_memory_batch_size'])
    max_bs = int(FASTOPIC_DEFAULTS['max_low_memory_batch_size'])
    if n_docs <= min_bs:
        return int(n_docs)
    candidate = max(min_bs, int(0.2 * n_docs))
    return int(min(candidate, max_bs))


def build_doc_embeddings_from_bow(bow_csr, word_emb, normalize=True, eps=1e-12, chunk_size=20000):
    n_docs = bow_csr.shape[0]
    emb_dim = word_emb.shape[1]
    out = np.zeros((n_docs, emb_dim), dtype=np.float32)

    lengths = np.asarray(bow_csr.sum(axis=1), dtype=np.float32).reshape(-1, 1)
    lengths[lengths == 0] = 1.0

    for start in range(0, n_docs, chunk_size):
        end = min(start + chunk_size, n_docs)
        part = bow_csr[start:end].dot(word_emb)
        out[start:end] = np.asarray(part, dtype=np.float32) / lengths[start:end]

    if normalize:
        norms = np.linalg.norm(out, axis=1, keepdims=True) + eps
        out = out / norms

    return out


class FixedPreprocess:
    def __init__(self, train_bow, vocab):
        self._train_bow = train_bow
        self._vocab = vocab

    def preprocess(self, docs):
        if len(docs) != self._train_bow.shape[0]:
            raise ValueError('docs length must match train_bow rows')
        return {'train_bow': self._train_bow, 'vocab': self._vocab}


class PassthroughDocEmbedder:
    def encode(self, docs, show_progress_bar=False, normalize_embeddings=False):
        raise RuntimeError('encode should not be called when preset_doc_embeddings is provided')


def infer_theta_from_doc_embeddings(model, doc_embeddings_np):
    doc_embeddings = torch.as_tensor(doc_embeddings_np, dtype=torch.float32)
    if not model.low_memory:
        doc_embeddings = doc_embeddings.to(model.device)
    return np.asarray(model.transform(doc_embeddings=doc_embeddings), dtype=np.float32)


OUTPUT_DIR: /kaggle/working/Sortie_mat/output/kaggle/FASTopic
20NG cfg: None
AGNews cfg: None
IMDB cfg: None
YahooAnswer cfg: None


def build_bow_avg_doc_embeddings(data_dir, train_bow, test_bow, vocab_size):
    word_emb = load_word_embeddings(data_dir / 'word_embeddings.npz', vocab_size=vocab_size)
    train_doc_emb = build_doc_embeddings_from_bow(
        train_bow,
        word_emb,
        normalize=FASTOPIC_DEFAULTS['normalize_embeddings'],
    )
    test_doc_emb = build_doc_embeddings_from_bow(
        test_bow,
        word_emb,
        normalize=FASTOPIC_DEFAULTS['normalize_embeddings'],
    )
    return train_doc_emb, test_doc_emb


def make_fastopic_model(K, preprocess, doc_embed_model, device, low_memory_batch_size, dt_alpha, tw_alpha, theta_temp):
    return FASTopic(
        num_topics=K,
        preprocess=preprocess,
        doc_embed_model=doc_embed_model,
        device=device,
        low_memory=bool(FASTOPIC_DEFAULTS['low_memory']),
        low_memory_batch_size=low_memory_batch_size if FASTOPIC_DEFAULTS['low_memory'] else None,
        normalize_embeddings=bool(FASTOPIC_DEFAULTS['normalize_embeddings']),
        DT_alpha=dt_alpha,
        TW_alpha=tw_alpha,
        theta_temp=theta_temp,
        verbose=False,
    )


def train_one_fastopic(dataset, K, seed=42):
    if dataset not in DATASET_DIRS:
        raise KeyError(f'Dataset {dataset} not found. Available: {list(DATASET_DIRS.keys())}')

    set_seed(seed)
    data_dir = DATASET_DIRS[dataset]
    cfg = load_cfg(CONFIGS.get(dataset))

    # Keep FASTopic-specific defaults; allow optional overrides via custom keys.
    epochs = int(cfg.get('fastopic_epochs', FASTOPIC_DEFAULTS['epochs']))
    learning_rate = float(cfg.get('fastopic_learning_rate', FASTOPIC_DEFAULTS['learning_rate']))

    train_bow = load_bow_csr(data_dir / 'train_bow.npz')
    test_bow = load_bow_csr(data_dir / 'test_bow.npz')
    vocab = load_vocab(data_dir / 'vocab.txt')
    vocab_size = train_bow.shape[1]

    preprocess = FixedPreprocess(train_bow=train_bow, vocab=vocab)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    train_docs = load_texts_if_exists(data_dir / 'train_texts.txt')
    test_docs = load_texts_if_exists(data_dir / 'test_texts.txt')

    selected_mode = resolve_doc_mode(dataset, train_docs)
    can_use_text = (
        train_docs is not None
        and test_docs is not None
        and len(train_docs) == train_bow.shape[0]
        and len(test_docs) == test_bow.shape[0]
    )
    use_text_mode = (selected_mode == 'text' and can_use_text)

    test_doc_emb = None
    if use_text_mode:
        docs_for_fit = train_docs
        preset_doc_embeddings = None
        doc_embed_model = FASTOPIC_DEFAULTS['doc_embed_model']
        avg_len = average_token_length(train_docs)
        print(f'[FASTopic] {dataset} K={K}: text mode (avg train tokens={avg_len:.1f})')
    else:
        train_doc_emb, test_doc_emb = build_bow_avg_doc_embeddings(data_dir, train_bow, test_bow, vocab_size)
        docs_for_fit = [''] * train_bow.shape[0]
        preset_doc_embeddings = train_doc_emb
        doc_embed_model = PassthroughDocEmbedder()
        print(f'[FASTopic] {dataset} K={K}: bow_avg mode')

    low_memory_batch_size = resolve_low_memory_batch_size(train_bow.shape[0])
    dt_alpha = float(cfg.get('fastopic_dt_alpha', DT_ALPHA_BY_DATASET.get(dataset, FASTOPIC_DEFAULTS['dt_alpha_default'])))
    tw_alpha = float(cfg.get('fastopic_tw_alpha', FASTOPIC_DEFAULTS['tw_alpha']))
    theta_temp = float(cfg.get('fastopic_theta_temp', FASTOPIC_DEFAULTS['theta_temp']))

    print(
        f"[FASTopic] {dataset} K={K} epochs={epochs} lr={learning_rate} "
        f"low_memory_batch_size={low_memory_batch_size} DT_alpha={dt_alpha}"
    )

    def _fit_current_setup():
        model = make_fastopic_model(
            K=K,
            preprocess=preprocess,
            doc_embed_model=doc_embed_model,
            device=device,
            low_memory_batch_size=low_memory_batch_size,
            dt_alpha=dt_alpha,
            tw_alpha=tw_alpha,
            theta_temp=theta_temp,
        )
        model.fit_transform(
            docs=docs_for_fit,
            epochs=epochs,
            learning_rate=learning_rate,
            preset_doc_embeddings=preset_doc_embeddings,
        )
        return model

    try:
        model = _fit_current_setup()
    except Exception as exc:
        if use_text_mode:
            # Robust fallback when sentence-transformer model download/runtime fails.
            print(f'[FASTopic] text mode failed for {dataset} K={K}: {repr(exc)}')
            print('[FASTopic] fallback to bow_avg mode')
            train_doc_emb, test_doc_emb = build_bow_avg_doc_embeddings(data_dir, train_bow, test_bow, vocab_size)
            docs_for_fit = [''] * train_bow.shape[0]
            preset_doc_embeddings = train_doc_emb
            doc_embed_model = PassthroughDocEmbedder()
            use_text_mode = False
            model = _fit_current_setup()
        else:
            raise

    beta = np.asarray(model.get_beta(), dtype=np.float32)
    train_theta = np.asarray(model.train_theta, dtype=np.float32)

    if use_text_mode:
        test_theta = np.asarray(model.transform(docs=test_docs), dtype=np.float32)
    else:
        test_theta = infer_theta_from_doc_embeddings(model, test_doc_emb)

    ds_out = OUTPUT_DIR / dataset
    ds_out.mkdir(parents=True, exist_ok=True)
    out_path = ds_out / f'{dataset}_FASTopic_K{K}_seed{seed}.mat'

    scipy.io.savemat(
        str(out_path),
        {
            'beta': beta,
            'train_theta': train_theta,
            'test_theta': test_theta,
            'traintheta': train_theta,
            'testtheta': test_theta,
        },
    )

    print('Saved:', out_path)
    print('beta', beta.shape, 'train_theta', train_theta.shape, 'test_theta', test_theta.shape)
    return out_path


In [20]:
def train_one_fastopic(dataset, K, seed=42):
    if dataset not in DATASET_DIRS:
        raise KeyError(f'Dataset {dataset} not found. Available: {list(DATASET_DIRS.keys())}')

    set_seed(seed)
    total_start = time.perf_counter()

    data_dir = DATASET_DIRS[dataset]
    cfg = load_cfg(CONFIGS.get(dataset))

    epochs = int(cfg.get('fastopic_epochs', FASTOPIC_DEFAULTS['epochs']))
    learning_rate = float(cfg.get('fastopic_learning_rate', FASTOPIC_DEFAULTS['learning_rate']))

    train_bow = load_bow_csr(data_dir / 'train_bow.npz')
    test_bow = load_bow_csr(data_dir / 'test_bow.npz')
    vocab = load_vocab(data_dir / 'vocab.txt')
    vocab_size = train_bow.shape[1]

    preprocess = FixedPreprocess(train_bow=train_bow, vocab=vocab)
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    train_docs = load_texts_if_exists(data_dir / 'train_texts.txt')
    test_docs = load_texts_if_exists(data_dir / 'test_texts.txt')

    selected_mode = resolve_doc_mode(dataset, train_docs)
    can_use_text = (
        train_docs is not None
        and test_docs is not None
        and len(train_docs) == train_bow.shape[0]
        and len(test_docs) == test_bow.shape[0]
    )
    use_text_mode = (selected_mode == 'text' and can_use_text)

    if use_text_mode:
        docs_for_fit = train_docs
        preset_doc_embeddings = None
        doc_embed_model = FASTOPIC_DEFAULTS['doc_embed_model']
        avg_len = average_token_length(train_docs)
        print(f'[FASTopic] {dataset} K={K}: text mode (avg train tokens={avg_len:.1f})')
    else:
        word_emb = load_word_embeddings(data_dir / 'word_embeddings.npz', vocab_size=vocab_size)
        train_doc_emb = build_doc_embeddings_from_bow(train_bow, word_emb, normalize=FASTOPIC_DEFAULTS['normalize_embeddings'])
        test_doc_emb = build_doc_embeddings_from_bow(test_bow, word_emb, normalize=FASTOPIC_DEFAULTS['normalize_embeddings'])
        docs_for_fit = [''] * train_bow.shape[0]
        preset_doc_embeddings = train_doc_emb
        doc_embed_model = PassthroughDocEmbedder()
        print(f'[FASTopic] {dataset} K={K}: bow_avg mode')

    low_memory_batch_size = resolve_low_memory_batch_size(train_bow.shape[0])
    dt_alpha = float(cfg.get('fastopic_dt_alpha', DT_ALPHA_BY_DATASET.get(dataset, FASTOPIC_DEFAULTS['dt_alpha_default'])))
    tw_alpha = float(cfg.get('fastopic_tw_alpha', FASTOPIC_DEFAULTS['tw_alpha']))
    theta_temp = float(cfg.get('fastopic_theta_temp', FASTOPIC_DEFAULTS['theta_temp']))

    print(f"[FASTopic] {dataset} K={K} epochs={epochs} lr={learning_rate} low_memory_batch_size={low_memory_batch_size} DT_alpha={dt_alpha}")

    model = FASTopic(
        num_topics=K,
        preprocess=preprocess,
        doc_embed_model=doc_embed_model,
        device=device,
        low_memory=bool(FASTOPIC_DEFAULTS['low_memory']),
        low_memory_batch_size=low_memory_batch_size if FASTOPIC_DEFAULTS['low_memory'] else None,
        normalize_embeddings=bool(FASTOPIC_DEFAULTS['normalize_embeddings']),
        DT_alpha=dt_alpha,
        TW_alpha=tw_alpha,
        theta_temp=theta_temp,
        verbose=False,
    )

    train_start = time.perf_counter()
    model.fit_transform(docs=docs_for_fit, epochs=epochs, learning_rate=learning_rate, preset_doc_embeddings=preset_doc_embeddings)
    train_seconds = time.perf_counter() - train_start

    beta = np.asarray(model.get_beta(), dtype=np.float32)
    train_theta = np.asarray(model.train_theta, dtype=np.float32)

    if use_text_mode:
        test_theta = np.asarray(model.transform(docs=test_docs), dtype=np.float32)
    else:
        test_theta = infer_theta_from_doc_embeddings(model, test_doc_emb)

    ds_out = OUTPUT_DIR / dataset
    ds_out.mkdir(parents=True, exist_ok=True)
    out_path = ds_out / f'{dataset}_FASTopic_K{K}_seed{seed}.mat'

    scipy.io.savemat(str(out_path), {'beta': beta, 'train_theta': train_theta, 'test_theta': test_theta, 'traintheta': train_theta, 'testtheta': test_theta})

    total_seconds = time.perf_counter() - total_start
    timing_row = {
        'model': 'FASTopic', 'dataset': dataset, 'K': int(K), 'seed': int(seed), 'device': device,
        'phase': 'train', 'train_seconds': float(train_seconds), 'total_seconds': float(total_seconds),
        'train_minutes': float(train_seconds / 60.0), 'total_minutes': float(total_seconds / 60.0),
        'mode': 'text' if use_text_mode else 'bow_avg',
    }

    timing_path = ds_out / f'{dataset}_FASTopic_K{K}_seed{seed}_timing.csv'
    pd.DataFrame([timing_row]).to_csv(timing_path, index=False)

    print('Saved:', out_path)
    print('Timing:', timing_path, f"| train={train_seconds:.2f}s total={total_seconds:.2f}s")

    return {'mat_path': str(out_path), 'timing_path': str(timing_path), 'timing': timing_row}


## 6) Run


In [21]:
training_time_rows = []

for dataset in TARGET_DATASETS:
    if dataset not in DATASET_DIRS:
        print('SKIP missing dataset:', dataset)
        continue
    for K in K_LIST:
        result = train_one_fastopic(dataset, K=K, seed=RANDOM_SEED)
        if isinstance(result, dict) and 'timing' in result:
            training_time_rows.append(result['timing'])

if training_time_rows:
    df_train_times = pd.DataFrame(training_time_rows).sort_values(['dataset', 'K']).reset_index(drop=True)
    display(df_train_times[['dataset', 'K', 'device', 'train_seconds', 'total_seconds', 'mode']])

    time_csv = OUTPUT_DIR / 'FASTopic_training_times.csv'
    df_train_times.to_csv(time_csv, index=False)
    print('Saved training times:', time_csv)
else:
    print('No training timing rows collected')


[FASTopic] 20NG K=20: text mode (avg train tokens=110.5)
[FASTopic] 20NG K=20 epochs=200 lr=0.002 low_memory_batch_size=2262 DT_alpha=8.0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Training FASTopic: 100%|██████████| 200/200 [02:34<00:00,  1.29it/s]


Saved: /kaggle/working/Sortie_mat/output/kaggle/FASTopic/20NG/20NG_FASTopic_K20_seed42.mat
Timing: /kaggle/working/Sortie_mat/output/kaggle/FASTopic/20NG/20NG_FASTopic_K20_seed42_timing.csv | train=165.27s total=171.63s
[FASTopic] 20NG K=50: text mode (avg train tokens=110.5)
[FASTopic] 20NG K=50 epochs=200 lr=0.002 low_memory_batch_size=2262 DT_alpha=8.0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

Saved: /kaggle/working/Sortie_mat/output/kaggle/FASTopic/YahooAnswer/YahooAnswer_FASTopic_K50_seed42.mat
Timing: /kaggle/working/Sortie_mat/output/kaggle/FASTopic/YahooAnswer/YahooAnswer_FASTopic_K50_seed42_timing.csv | train=148.89s total=149.07s
[FASTopic] YahooAnswer K=100: bow_avg mode
[FASTopic] YahooAnswer K=100 epochs=200 lr=0.002 low_memory_batch_size=2000 DT_alpha=12.0


Training FASTopic: 100%|██████████| 200/200 [02:32<00:00,  1.31it/s]


Saved: /kaggle/working/Sortie_mat/output/kaggle/FASTopic/YahooAnswer/YahooAnswer_FASTopic_K100_seed42.mat
Timing: /kaggle/working/Sortie_mat/output/kaggle/FASTopic/YahooAnswer/YahooAnswer_FASTopic_K100_seed42_timing.csv | train=152.39s total=152.58s


,dataset,K,device,train_seconds,total_seconds,mode
0,20NG,20,cuda,165.274803,171.634118,text
1,20NG,50,cuda,165.143862,171.517996,text
2,20NG,100,cuda,167.564222,174.050345,text
3,AGNews,20,cuda,132.181618,132.410216,bow_avg
4,AGNews,50,cuda,92.176522,92.310317,bow_avg
5,AGNews,100,cuda,94.114964,94.283610,bow_avg
6,IMDB,20,cuda,326.830232,347.899832,text
7,IMDB,50,cuda,327.131580,348.390479,text
8,IMDB,100,cuda,328.520936,349.572929,text
9,YahooAnswer,20,cuda,90.483272,90.814363,bow_avg


Saved training times: /kaggle/working/Sortie_mat/output/kaggle/FASTopic/FASTopic_training_times.csv


## 7) Metrics (C_V Gensim + TD + Purity + NMI) and topics files


In [22]:
import pandas as pd
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel


def topic_diversity_from_beta(beta, topn=15):
    all_words = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:topn]
        all_words.extend(top_idx.tolist())
    return len(set(all_words)) / max(1, len(all_words))


def purity_score(y_true, y_pred):
    contingency_matrix = metrics.cluster.contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)


def build_tokenized_texts_from_bow(train_bow, vocab, max_docs=10000, min_tokens=2):
    texts = []
    n_docs = min(train_bow.shape[0], max_docs)

    for i in range(n_docs):
        row = train_bow.getrow(i)
        idx = row.indices
        if len(idx) < min_tokens:
            continue
        words = [vocab[j] for j in idx if j < len(vocab)]
        if len(words) >= min_tokens:
            texts.append(words)

    return texts


def compute_cv_gensim(beta, vocab, texts, dictionary, topn=15):
    topics = []
    for k in range(beta.shape[0]):
        top_idx = np.argsort(beta[k])[::-1][:topn]
        topic_words = [
            vocab[i]
            for i in top_idx
            if i < len(vocab) and vocab[i] in dictionary.token2id
        ]
        if len(topic_words) >= 2:
            topics.append(topic_words)

    if not topics:
        return float('nan')

    cm = CoherenceModel(
        topics=topics,
        texts=texts,
        dictionary=dictionary,
        coherence='c_v',
    )
    return float(cm.get_coherence())


rows = []
TOPN = 15
TEXTS_MAX_DOCS = 10000

for dataset in TARGET_DATASETS:
    if dataset not in DATASET_DIRS:
        continue

    ds_out = OUTPUT_DIR / dataset
    data_dir = DATASET_DIRS[dataset]
    vocab = load_vocab(data_dir / 'vocab.txt')
    labels = np.loadtxt(data_dir / 'test_labels.txt', dtype=np.int32)

    train_bow_for_cv = load_bow_csr(data_dir / 'train_bow.npz')
    tokenized_texts = build_tokenized_texts_from_bow(
        train_bow_for_cv,
        vocab,
        max_docs=TEXTS_MAX_DOCS,
    )
    dictionary = Dictionary(tokenized_texts)
    print(f"[C_V] {dataset}: {len(tokenized_texts)} docs tokenises, {len(dictionary)} tokens")

    for K in K_LIST:
        mat_path = ds_out / f'{dataset}_FASTopic_K{K}_seed{RANDOM_SEED}.mat'
        if not mat_path.exists():
            continue

        loaded = scipy.io.loadmat(str(mat_path))
        beta = loaded['beta']
        test_theta = loaded['test_theta']

        n_eval = min(len(labels), test_theta.shape[0])
        y_true = labels[:n_eval]
        y_pred = np.argmax(test_theta[:n_eval], axis=1)

        cv_score = compute_cv_gensim(beta, vocab, tokenized_texts, dictionary, topn=TOPN)

        rows.append({
            'model': 'FASTopic',
            'dataset': dataset,
            'K': K,
            'C_V': cv_score,
            'Purity': float(purity_score(y_true, y_pred)),
            'NMI': float(metrics.cluster.normalized_mutual_info_score(y_true, y_pred)),
            'TD_top15': float(topic_diversity_from_beta(beta, topn=15)),
            'TD_top25': float(topic_diversity_from_beta(beta, topn=25)),
        })

        txt_path = ds_out / f'topics_for_cv_{dataset}_FASTopic_K{K}_seed{RANDOM_SEED}.txt'
        with open(txt_path, 'w', encoding='utf-8') as f:
            for k in range(beta.shape[0]):
                top_idx = np.argsort(beta[k])[::-1][:TOPN]
                f.write(' '.join(vocab[i] for i in top_idx) + '\n')
        print(f"Saved topics + metrics input: {txt_path} | C_V={cv_score:.4f}")

if rows:
    df = pd.DataFrame(rows).sort_values(['dataset', 'K']).reset_index(drop=True)
    display(df)

    csv_path = OUTPUT_DIR / 'FASTopic_metrics_CV_TD_Purity_NMI.csv'
    df.to_csv(csv_path, index=False)
    print('Saved:', csv_path)

    pivot_cv = df.pivot(index='dataset', columns='K', values='C_V')
    print('\nC_V (Gensim c_v)')
    display(pivot_cv.round(4))
else:
    print('No FASTopic .mat found yet')

print('\\nFiles in OUTPUT_DIR:')
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        print('-', p.relative_to(OUTPUT_DIR))


[C_V] 20NG: 10000 docs tokenises, 5000 tokens
Saved topics + metrics input: /kaggle/working/Sortie_mat/output/kaggle/FASTopic/20NG/topics_for_cv_20NG_FASTopic_K20_seed42.txt | C_V=0.4714
Saved topics + metrics input: /kaggle/working/Sortie_mat/output/kaggle/FASTopic/20NG/topics_for_cv_20NG_FASTopic_K50_seed42.txt | C_V=0.5082
Saved topics + metrics input: /kaggle/working/Sortie_mat/output/kaggle/FASTopic/20NG/topics_for_cv_20NG_FASTopic_K100_seed42.txt | C_V=0.4588
[C_V] AGNews: 9999 docs tokenises, 4999 tokens
Saved topics + metrics input: /kaggle/working/Sortie_mat/output/kaggle/FASTopic/AGNews/topics_for_cv_AGNews_FASTopic_K20_seed42.txt | C_V=0.5331
Saved topics + metrics input: /kaggle/working/Sortie_mat/output/kaggle/FASTopic/AGNews/topics_for_cv_AGNews_FASTopic_K50_seed42.txt | C_V=0.4987
Saved topics + metrics input: /kaggle/working/Sortie_mat/output/kaggle/FASTopic/AGNews/topics_for_cv_AGNews_FASTopic_K100_seed42.txt | C_V=0.4852
[C_V] IMDB: 10000 docs tokenises, 5000 tokens
S

,model,dataset,K,C_V,Purity,NMI,TD_top15,TD_top25
0,FASTopic,20NG,20,0.471387,0.490441,0.518917,0.980000,0.9680
1,FASTopic,20NG,50,0.508166,0.597982,0.512810,0.949333,0.9080
2,FASTopic,20NG,100,0.458832,0.599973,0.484845,0.908000,0.8416
3,FASTopic,AGNews,20,0.533102,0.821600,0.404296,0.956667,0.9220
4,FASTopic,AGNews,50,0.498673,0.828000,0.344750,0.910667,0.8512
5,FASTopic,AGNews,100,0.485238,0.824800,0.315858,0.852667,0.7992
6,FASTopic,IMDB,20,0.398752,0.668640,0.059044,0.950000,0.9100
7,FASTopic,IMDB,50,0.398406,0.661200,0.040708,0.904000,0.8456
8,FASTopic,IMDB,100,0.372549,0.668320,0.036824,0.853333,0.7856
9,FASTopic,YahooAnswer,20,0.564440,0.529200,0.312266,0.960000,0.9340


Saved: /kaggle/working/Sortie_mat/output/kaggle/FASTopic/FASTopic_metrics_CV_TD_Purity_NMI.csv

C_V (Gensim c_v)


K,20,50,100
dataset,,,
20NG,0.4714,0.5082,0.4588
AGNews,0.5331,0.4987,0.4852
IMDB,0.3988,0.3984,0.3725
YahooAnswer,0.5644,0.5395,0.5148


\nFiles in OUTPUT_DIR:
- 20NG/20NG_FASTopic_K100_seed42.mat
- 20NG/20NG_FASTopic_K100_seed42_timing.csv
- 20NG/20NG_FASTopic_K20_seed42.mat
- 20NG/20NG_FASTopic_K20_seed42_timing.csv
- 20NG/20NG_FASTopic_K50_seed42.mat
- 20NG/20NG_FASTopic_K50_seed42_timing.csv
- 20NG/topics_for_cv_20NG_FASTopic_K100_seed42.txt
- 20NG/topics_for_cv_20NG_FASTopic_K20_seed42.txt
- 20NG/topics_for_cv_20NG_FASTopic_K50_seed42.txt
- AGNews/AGNews_FASTopic_K100_seed42.mat
- AGNews/AGNews_FASTopic_K100_seed42_timing.csv
- AGNews/AGNews_FASTopic_K20_seed42.mat
- AGNews/AGNews_FASTopic_K20_seed42_timing.csv
- AGNews/AGNews_FASTopic_K50_seed42.mat
- AGNews/AGNews_FASTopic_K50_seed42_timing.csv
- AGNews/topics_for_cv_AGNews_FASTopic_K100_seed42.txt
- AGNews/topics_for_cv_AGNews_FASTopic_K20_seed42.txt
- AGNews/topics_for_cv_AGNews_FASTopic_K50_seed42.txt
- FASTopic_metrics_CV_TD_Purity_NMI.csv
- FASTopic_metrics_TD_Purity_NMI.csv
- FASTopic_training_times.csv
- IMDB/IMDB_FASTopic_K100_seed42.mat
- IMDB/IMDB_FASTo